In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()


#preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import mode


# pipelines
from sklearn.pipeline import Pipeline



# Customer Data

# Tips

In [ ]:
# Read in the data found in "best_one_ever_database.csv" and take a look at the head and info.
data = pd.read_csv("best_one_ever_database.csv", index_col = 'id')

In [ ]:
data.head()

In [ ]:
data.info()

# Cleaning The Data

In [ ]:
# drop the name columns using pandas.drop()

X = data.drop(labels = ['first_name','last_name'], axis = 1)

In [ ]:
X.info()

### Transform string columns into useful features

In [ ]:
def strip_dollar(x):
    return x[1:]

In [ ]:
# apply the function to the column and assign it back to the column (it does not work inplace)
X.sales = X.sales.apply(strip_dollar)

# cast the column to a floating point type - this is very important, otherwise it will be
# an object type column that we cannot do arithmetic on the column
X.sales = X.sales.astype('float32')

In [ ]:
X.head()

### Your Turn

In [ ]:
# define functions to apply to the dataframe
def strip_emails(x):
    at = x.find('@')
    return x[at+1:]

test_email = 'thisismymail@gmail.com'
print(strip_emails(test_email))

In [ ]:
# apply the function to the column and assign it back to the column (it does not work inplace)
X.email = X.email.apply(strip_emails)
X.head()

# Is the email column going to be worth it?

In [ ]:
# how many unique domains are there?
counts = X.email.value_counts()

In [ ]:
print(counts.values)

In [ ]:
# we make sure to copy it over and save it for later.
emails = X.email.copy()
X.drop('email', axis=1, inplace = True)

In [ ]:
X.info()

## Splitting the IP Address

In [ ]:
X[['first_ip','second_ip','third_ip','fourth_ip']] = X.ip_address.str.split(pat=".", expand=True)

In [ ]:
# now we cast the columns as floats
X[['first_ip','second_ip','third_ip','fourth_ip']] = X[['first_ip','second_ip','third_ip','fourth_ip']].astype('float32')
# we also drop the original column
X.drop('ip_address', axis=1, inplace=True)

In [ ]:
X.info()

# One Hot Encoding

In [ ]:
X.gender.value_counts()

In [ ]:
dumbdumbs = pd.get_dummies(X['gender'])
X= pd.concat((X,dumbdumbs), axis = 1)
X.drop(['gender'], axis = 1, inplace=True)
X.head()

In [ ]:
X.info()

# Data Preprocessing Stage 2 - Missing values

In [ ]:
from sklearn.impute import SimpleImputer
imp = SimpleImputer(missing_values=np.nan, strategy='mean')

In [ ]:
X_ = imp.fit_transform(X)

In [ ]:
X= pd.DataFrame(X_, columns=X.columns)

In [ ]:
X.info()

In [ ]:
X.head()

# Clustering

## K-means
#### Note:

In [ ]:
# clustering
from sklearn.cluster import DBSCAN, KMeans

# clustering metrics
from sklearn.metrics import silhouette_score

In [ ]:
# scale your data with a scaler of your choice
ss = StandardScaler()
x_scaled = ss.fit_transform(X)

In [ ]:
# random guess for k
kmeans = KMeans(n_clusters = 10)
kmeans.fit(x_scaled)

In [ ]:
labels = kmeans.predict(x_scaled)
print(silhouette_score(x_scaled, labels))

In [ ]:
for i in range(2,20):
    kmeans = KMeans(n_clusters = i)
    kmeans.fit(x_scaled)
    labels = kmeans.predict(x_scaled)
    print(f"The number of clusters is {i} and the Silhouette score is {silhouette_score(x_scaled, labels)}")

### Cluster Discussion

In [ ]:
# fit a kmeans cluster with desired number of components
kmeans = KMeans(3).fit(x_scaled)

In [ ]:
X.shape

In [ ]:
# look for the centers of your clusters.  
# Hint: your kmeans object has an attribute you'd want to use
centers = kmeans.cluster_centers_
print(kmeans.cluster_centers_.shape)
print(kmeans.cluster_centers_)


In [ ]:
# my gift to you
def plot_centroids(centers):
    pd.DataFrame(centers, columns = X.columns).plot(kind = 'bar', figsize = (12,6))
    plt.legend(loc='best', bbox_to_anchor=(0.8, 0.1, 0.5, 0.5)); # bbox is (x,y, width, height)


In [ ]:
plot_centroids(centers)

# Examine your centroids

# DB-Scan

In [ ]:
# run dbscan 
# pick an eps -- how might we do this? what range are your features in?
# Your eps should be relevant to the feature space you exist in
# pick min_samples

dbscan = DBSCAN(eps=3.0, min_samples=5).fit(x_scaled)

#### ok, now you fit a dbscan.

In [ ]:
dbscan.labels_

In [ ]:
dbscan.labels_.shape

In [ ]:
dbscan.components_.shape

In [ ]:
dbscan.core_sample_indices_

In [ ]:
dbscan.core_sample_indices_.shape

## Calculate the Silhouette Score
#### NOTE
### Number of clusters?
### Goal

In [ ]:
uniques, counts = np.unique(dbscan.labels_, return_counts=True)
print(uniques)
print(counts)

In [ ]:
dbscan.core_sample_indices_.shape

In [ ]:
dbscan.labels_!=-1

In [ ]:
x_scaled.shape

In [ ]:
## remove the -1's from the labels.
## you also will need to remove the corresponding x_scaled samples so you can compute the silhouette score.
# you can use numpy boolean indexing masks for this.

labels = dbscan.labels_[dbscan.labels_!=-1]
x_points = x_scaled[dbscan.labels_!=-1]

In [ ]:
labels.shape

In [ ]:
x_points.shape

In [ ]:
## ok if you have the labels and x's without -1's you can actually computer the Silhouette score now
silhouette_score(x_points, labels)

In [ ]:
## how many clusters are there though?
## you need to figure out how many unique labels there are, and also it would be nice to know
## how many points are in each cluster

In [ ]:
#  https://www.kite.com/python/answers/how-to-count-frequency-of-unique-values-in-a-numpy-array-in-python

uniques, counts = np.unique(labels, return_counts=True)

In [ ]:
print(uniques)

In [ ]:
print(np.asarray((uniques,counts)).T)

In [ ]:
num_clusters = len(np.unique(labels))
print(f"The number of clusters is {num_clusters} and the Silhouette score is {silhouette_score(x_points, labels)}")

### How does it compare?
### Next we'd like to look at the centroids

In [ ]:
labels.shape

In [ ]:
centers = []
for label in uniques:
    centers.append(x_points[labels==label].mean(axis=0))

dbcenters = np.array(centers)

In [ ]:
dbcenters

In [ ]:
plot_centroids(dbcenters)

# How do your centroids compare?

# PCA

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
# make a PCA object.
# fit your PCA object on the scaled data
# look at the PCA shape , is it what you expected?


In [ ]:
def display_scree_plot(pca):
    '''Display a scree plot for the pca'''
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    
    scree = pca.explained_variance_ratio_*100
    ax.bar(np.arange(len(scree))+1, scree)
    ax.plot(np.arange(len(scree))+1, scree.cumsum(),c="red",marker='o')
    ax.set_xlabel("Principal components")
    ax.set_ylabel("Percentage explained variance")
    ax.set_title("Scree plot")
    return ax

In [ ]:
# plot your scree plot


# The 'elbow' method of cumsum plots

# Your answer here:

# PCA plots

In [ ]:
# transform the scaled data with PCA
x_sca_pca = pca.transform(x_scaled)

In [ ]:
# perform a scatter plot of the PCA data
# you'll want to use the first and second PCA components to plot.
fig, ax = plt.subplots(figsize=(10,6))


In [ ]:
# get labels from K-means
# note we should predict on the same data we fit K-means with. So if we fit with data "x_scaled" we need to be sure to predict on that data.
# it would be a mistake to make predictions on the PCA transformed data now, because we did not fit the kmeans object on the PCA data!


In [ ]:
fig, ax = plt.subplots(figsize=(10,6));

# plot your datapoints again, this time with colors for labels.  I gave you some helper code below
ax.scatter(x_sca_pca[:,0],x_sca_pca[:,1], c=labels, cmap='viridis');

#### What do you think of the results?

## PCA Components examined

In [ ]:
def plot_pca_comps(pca_item, num_dims, data):
    
    variance_ratios = pca_item.explained_variance_ratio_
    components = pd.DataFrame(pca_item.components_[:num_dims], columns = data.columns)
    
    fig, ax = plt.subplots(figsize = (14,8))
    
    components.plot(ax=ax, kind = 'bar');
    ax.set_ylabel("Feature Weights")
    dims = [f'Dimension {i}' for i in range(num_dims)]
    ax.set_xticklabels(labels =dims,rotation=0)
    
    # Display the explained variance ratios
    for i, ev in enumerate(pca_item.explained_variance_ratio_[:num_dims]):
        ax.text(i-0.40, ax.get_ylim()[1] + 0.05, f"Explained Variance\n {np.round(ev,4)}")

    ax.legend(loc='best', bbox_to_anchor=(0.75, 0.1, 0.5, 0.5)) # bbox is (x,y, width, height)
    


In [ ]:
# pass PCA, # of dimensions to plot, X the data )
plot_pca_comps(pca, 2, X)

# What do the loadings tell us?

#### Your answer here

## feel free to look at more PCA loadings, extend the graph to all 6 components.

## Clustering on PCA transformed data

In [ ]:
## do the work here



In [ ]:

# plot your results here
fig, ax = plt.subplots(figsize=(10,6));
ax.scatter(pca_scaled_data[:,0],pca_scaled_data[:,1], c=labels, cmap='viridis');

In [ ]:
# In order to look at the centroids we need to inverse them back into the scaled space.
centers =   #your code here

In [ ]:
plot_centroids(centers)

### Can we learn clusters on the PCA data effectively? At least in this case